In [1]:
# 1. IMPORTS & DATA LOADING
# ==========================================
import pandas as pd
import numpy as np
import datetime
from datetime import datetime

# Load datasets
try:
    df = pd.read_csv('201 Bills and Payments.csv')
    pc = pd.read_csv('All Encounters 2026-01-01 17_20_08.csv')
    charges = pd.read_csv('260101 Charges Report.csv')

    # Drop PII for safety
    if 'Patient ID' in charges.columns:
        charges = charges.drop(columns=['Patient ID', 'DOB'])

    print("Data Loaded Successfully.")
except FileNotFoundError as e:
    print(f"CRITICAL ERROR: {e}")
    print("Please ensure all CSV files are in the folder.")

master_start = pd.to_datetime('2025-10-01')
master_end = pd.to_datetime('2025-12-31')

CRITICAL ERROR: [Errno 2] No such file or directory: 'All Encounters 2026-01-01 17_20_08.csv'
Please ensure all CSV files are in the folder.


In [2]:
# 2. FINANCIAL TRUTH (Calculate Realization Rate FIRST)
# ==========================================
# We process charges first so we can use the Realization Rate
# to accurately predict revenue for the Clinical/Encounter data.

# A. Clean Currency Helper
def clean_currency(x):
    if isinstance(x, str):
        x = x.replace('$', '').replace(',', '').strip()
        if '(' in x and ')' in x: x = '-' + x.replace('(', '').replace(')', '')
    return pd.to_numeric(x, errors='coerce')

# B. Clean Money Columns in Charges
money_cols = [
    'Service Charge Amount', 'Allowed Amount', 'Pri Ins Insurance Contract Adjustment',
    'Sec Ins Insurance Contract Adjustment', 'Other Ins Insurance Contract Adjustment',
    'Pri Ins Insurance Payment', 'Sec Ins Insurance Payment',
    'Other Ins Insurance Payment', 'Pat Payment Amount'
]

for col in money_cols:
    if col in charges.columns:
        charges[col] = charges[col].apply(clean_currency).fillna(0)

# C. Calculate "True" Allowed Amount (Billed - Write-offs)
charges['Total_Contract_Adj'] = (
    charges['Pri Ins Insurance Contract Adjustment'] +
    charges['Sec Ins Insurance Contract Adjustment'] +
    charges['Other Ins Insurance Contract Adjustment']
)
charges['Calculated_Allowed'] = charges['Service Charge Amount'] - charges['Total_Contract_Adj']

# D. Calculate The Metric
total_billed = charges['Service Charge Amount'].sum()
total_allowed = charges['Calculated_Allowed'].sum()

# THE DYNAMIC FACTOR
# If total_billed is 0, default to 0.40 (40%) to prevent errors
if total_billed > 0:
    REALIZATION_FACTOR = total_allowed / total_billed
else:
    REALIZATION_FACTOR = 0.40

print(f"--- DYNAMIC CONFIGURATION ---")
print(f"Calculated Realization Rate: {REALIZATION_FACTOR*100:.2f}%")

NameError: name 'charges' is not defined

In [ ]:
# ==========================================
# 3. REVENUE REPORT (Using Dynamic Rate)
# ==========================================

# --- PART A: PROCESS CLINICAL (PC) DATA ---
pc = pc.rename(columns={'Rendering Provider': 'Provider',
                        'Procedure Codes with Modifiers': 'Services',
                        'Total Charge Amount': 'Total Charges'})

# Clean Billed Charges
pc['Billed'] = pc['Total Charges'].astype(str).str.replace(r'[$,]', '', regex=True).astype(float)

pc['Fees'] = pc['Billed'] * REALIZATION_FACTOR

# Normalize Providers
provider_map_pc = {
    'Jenks, Anne ': 'Anne Jenks',
    'IRVIN, EHRIN ': 'Ehrin Irvin',
    'SUGGS, SARAH ': 'Sarah Suggs',
    'REDD, DAVID ': 'Anne Jenks'
}
pc['Provider'] = pc['Provider'].replace(provider_map_pc)
pc['Date of Service'] = pd.to_datetime(pc['Date of Service'])

# Prepare Clinical Stack
pc_clean = pc[['Date of Service', 'Provider', 'Fees']].copy()
pc_clean['Revenue Type'] = 'Clinical'

# --- PART B: PROCESS DBQ/IMO (DF) DATA ---
# (This section uses fixed fee schedules, so Realization Rate doesn't apply here)
df = df.rename(columns={'Unnamed: 0': '201 ID', 'Sarah Suggs ': 'Provider'})

df['Date of Service'] = df['Date of Service'].ffill()
df['Date of Service'] = pd.to_datetime(df['Date of Service'], errors='coerce')
df = df.dropna(subset=['Date of Service']).fillna('0')

df['Provider'] = df['Provider'].str.strip().replace({
    '0': 'Anne Jenks', 'sArah Suggs': 'Sarah Suggs',
    'SArah Suggs': 'Sarah Suggs', 'Sarah Suggs': 'Sarah Suggs'
})

# Clean Calculation Columns
for col in ['Focused DBQs', 'Routine IMOs', 'Gen Med DBQs', 'No Show', 'TBI']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

# Mappings & Fees
dbq_map = {'Visual': '0', 'r-tbi': '0', '1--5': '1-5', 'bp check': '0', 'nan': '0', '': '0'}
imo_map = {'7/9': '7-9', '1/3': '1-3', '0': '0', '1-5': '1-3', '1': '1-3', '4': '4-6', '7': '7-9', '6-10': '4-6'}
gen_med_map = {'0': '0', 'BP CHECK': '0', '11-13': '11-15', '1-6': '1-5', '7': '6-10', '6': '6-10', '??': '0'}

df['Focused DBQs'] = df['Focused DBQs'].replace(dbq_map)
df['Routine IMOs'] = df['Routine IMOs'].replace(imo_map)
df['Gen Med DBQs'] = df['Gen Med DBQs'].replace(gen_med_map)
df.loc[df['No Show'] == '1', ['Focused DBQs', 'Gen Med DBQs', 'Routine IMOs']] = '0'

fees = {
    'GM_DBQs': {'0': 0, '1-5': 250, '6-10': 330, '11-15': 410, '16+': 550},
    'Focused': {'0': 0, '1-5': 200, '6-10': 280, '11-15': 370, '16+': 480},
    'TBI': 250,
    'IMO': {'0': 0, '1-3': 150, '4-6': 260, '7-9': 330, '10-12': 350, '13-15': 465, '16-18': 525, '19+': 600},
    'No_Show': {'0': 0, '1': 60}
}


def fee_calc(row):
    f_fee = fees['Focused'].get(row['Focused DBQs'], 0)
    imo_fee = fees['IMO'].get(row['Routine IMOs'], 0)
    gmed_fee = fees['GM_DBQs'].get(row['Gen Med DBQs'], 0)
    noshow_fee = fees['No_Show'].get(row['No Show'], 0)
    tbi_fee = 250 if row['TBI'] not in ['0', 'nan', '', 'no', 'false'] else 0
    return f_fee + imo_fee + gmed_fee + noshow_fee + tbi_fee


df['Fees'] = df.apply(fee_calc, axis=1)
df_clean = df[['Date of Service', 'Provider', 'Fees']].copy()
df_clean['Revenue Type'] = 'DBQ/IMO'

# --- PART C: MASTER REPORT ---
master_df = pd.concat([pc_clean, df_clean], ignore_index=True)
master_df = master_df.sort_values('Date of Service')
master_df['RVUs'] = master_df['Fees'] / 80

# Filter Date Range (Last 90 Days)
end_date = master_end
start_date = master_start
date_filtered_df = master_df[master_df['Date of Service'].between(start_date, end_date)]

report = date_filtered_df.groupby(['Provider']).agg({
    'Date of Service': 'count',
    'Fees': 'sum',
    'RVUs': 'sum'
}).reset_index().rename(columns={'Date of Service': 'Visits'})

print("\n--- MASTER REVENUE REPORT (Clinical Adjusted by Realization Rate) ---")
print(f'Starting {start_date} to {end_date}')

# Define specific formatting for each column
formatters = {
    'Fees': '${:,.2f}'.format,   # Currency format
    'RVUs': '{:,.0f}'.format,    # Simple float format (No $)
    'Visits': '{:.0f}'.format    # Integer format (No decimals)
}

# Apply the formatters
print(report.to_string(index=False, formatters=formatters))

print("-" * 50)
print(f"TOTAL ESTIMATED REVENUE: ${report['Fees'].sum():,.2f}")
print(f"TOTAL RVUs GENERATED:    {report['RVUs'].sum():,.2f}")


--- MASTER REVENUE REPORT (Clinical Adjusted by Realization Rate) ---
Starting 2025-10-01 00:00:00 to 2025-12-31 00:00:00
    Provider Visits       Fees RVUs
  Anne Jenks    247 $40,827.82  510
 Ehrin Irvin    656 $72,643.72  908
Heather Mayo    187 $57,820.00  723
 Sarah Suggs    190 $57,299.74  716
--------------------------------------------------
TOTAL ESTIMATED REVENUE: $228,591.28
TOTAL RVUs GENERATED:    2,857.39


C:\Users\jenks\AppData\Local\Temp\ipykernel_49940\3302029940.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date of Service'] = pd.to_datetime(df['Date of Service'], errors='coerce')


In [ ]:
# ==========================================
# 11. BONUS REPORT: CONTRACT RVU CALCULATOR & COUNTS
# ==========================================

# --- A. DEFINE THE BONUS LOGIC ---
def calculate_bonus(rvus):
    if rvus < 688:
        return 0
    elif rvus <= 735:
        return (rvus - 688) * 25
    elif rvus <= 827:
        return 1150 + (rvus - 735) * 30
    else:
        return 3910 + (rvus - 827) * 35


# --- B. CLINICAL RULES ENGINE (Returns RVU + Category Name) ---
def get_clinical_data(row):
    services = str(row['Services']).upper()
    charges = row['Total Charges']

    # 1. Functional Medicine
    if "FUNCTIONAL" in services and "INITIAL" in services:
        return 3.75, "Func Med Initial"
    if "FUNCTIONAL" in services:
        return 2.50, "Func Med Subsequent"

    # 2. Nexus Letter
    if "NEXUS" in services or charges == 200.00:
        return 2.5, "Nexus Letter"

    # 3. DOT Physical
    if "DOT" in services or ("PHYSICAL" in services and "DOT" in str(row).upper()):
        return 1.0, "DOT Physical"

    # 4. Primary Care Visit (1.3 RVUs)
    import re
    if re.search(r'992[01][1-5]', services): return 1.3, "Primary Care Visit"
    if re.search(r'993[89][0-9]', services): return 1.3, "Primary Care Visit"
    if "ANNUAL" in services or "PHYSICAL" in services: return 1.3, "Primary Care Visit"

    # 5. Vaccine Only
    if re.search(r'90[4-7][0-9]{2}', services) or "VACCINE" in services or "ADMIN" in services:
        return 0.5, "Vaccine Only"

    return 0.0, "Other/Unclassified"


# --- C. VA EXAM RULES ENGINE ---
def get_va_data(row):
    # CHECK FOR NO SHOW FIRST
    if str(row['No Show']).strip() == '1':
        return 0.75, "No Show"

    imo_str = str(row['Routine IMOs']).strip()

    if imo_str in ['0', 'nan', '']:
        return 2.5, "VA Exam (0 IMOs)"

    if imo_str in ['1-3', '1-5', '1', '2', '3'] or (imo_str[0] in ['1', '2', '3'] and len(imo_str) == 1):
        return 4.5, "VA Exam (1-3 IMOs)"

    return 6.5, "VA Exam (4+ IMOs)"


# --- D. APPLY LOGIC ---

# 1. Clinical Data
pc_bonus = pc.copy()
pc_bonus[['Contract_RVUs', 'Category']] = pc_bonus.apply(
    lambda x: pd.Series(get_clinical_data(x)), axis=1
)

# 2. VA Data
df_bonus = df.copy()
df_bonus[['Contract_RVUs', 'Category']] = df_bonus.apply(
    lambda x: pd.Series(get_va_data(x)), axis=1
)

# --- E. AGGREGATE ---

# Stack Data
cols = ['Provider', 'Date of Service', 'Contract_RVUs', 'Category']
master_bonus = pd.concat([pc_bonus[cols], df_bonus[cols]], ignore_index=True)

# Filter Date Range
end_date = master_end
start_date = master_start
master_bonus = master_bonus[master_bonus['Date of Service'].between(start_date, end_date)]

# --- F. GENERATE REPORTS ---

# REPORT 1: HIGH LEVEL BONUS PAYOUT
bonus_summary = master_bonus.groupby('Provider')['Contract_RVUs'].sum().reset_index()
bonus_summary['Bonus_Payout'] = bonus_summary['Contract_RVUs'].apply(calculate_bonus)

print("\n" + "=" * 60)
print("             QUARTERLY PROVIDER BONUS REPORT")
print(f"             Period: {start_date.date()} to {end_date.date()}")
print("=" * 60)
print(f"{'Provider':<20} | {'Total RVUs':<12} | {'Bonus Payout':<15}")
print("-" * 60)
for _, row in bonus_summary.iterrows():
    print(f"{row['Provider']:<20} | {row['Contract_RVUs']:<12.2f} | ${row['Bonus_Payout']:<15,.2f}")
print("-" * 60)
print(f"TOTAL PAYOUT: ${bonus_summary['Bonus_Payout'].sum():,.2f}")

# REPORT 2: DETAILED COUNTS WITH TOTALS
# 1. Create the Pivot Table
counts_pivot = pd.crosstab(master_bonus['Category'], master_bonus['Provider'])

# 2. Add Horizontal Total (Row Sum)
counts_pivot['Total'] = counts_pivot.sum(axis=1)

# 3. Add RVU Reference Column (Insert at position 0)
rvu_ref = master_bonus.groupby('Category')['Contract_RVUs'].max()
counts_pivot.insert(0, 'RVU Val', rvu_ref)

# 4. Add Vertical Total (Column Sum)
# We calculate sum of all columns (numeric only handles the pivot data)
total_row = counts_pivot.sum(numeric_only=True)
total_row.name = 'TOTAL VOLUME'
# We don't want to sum the 'RVU Val' column, it makes no sense, so set to NaN or blank
total_row['RVU Val'] = 0

# Append the total row
counts_pivot = pd.concat([counts_pivot, total_row.to_frame().T])

print("\n\n" + "=" * 60)
print("             DETAILED VOLUME BREAKDOWN (Counts)")
print("=" * 60)
# Format the table: Replace NaNs with '-' and ensure integers for counts
# We use a custom formatter to keep the table clean
print(
    counts_pivot.to_string(na_rep='-', float_format=lambda x: '{:.0f}'.format(x) if x % 1 == 0 else '{:.2f}'.format(x)))
print("-" * 60)


             QUARTERLY PROVIDER BONUS REPORT
             Period: 2025-10-01 to 2025-12-31
Provider             | Total RVUs   | Bonus Payout   
------------------------------------------------------------
Anne Jenks           | 408.35       | $0.00           
Ehrin Irvin          | 731.00       | $1,075.00       
Heather Mayo         | 738.50       | $1,255.00       
Sarah Suggs          | 724.10       | $902.50         
------------------------------------------------------------
TOTAL PAYOUT: $3,232.50


             DETAILED VOLUME BREAKDOWN (Counts)
Provider            RVU Val  Anne Jenks  Ehrin Irvin  Heather Mayo  Sarah Suggs  Total
No Show                0.75           1            0            20           10     31
Other/Unclassified        0          18           90             0            2    110
Primary Care Visit     1.30         182          560             0           22    764
VA Exam (0 IMOs)       2.50          12            0            52           49    113
VA 

In [ ]:
# 3. CHARGES ANALYSIS: CLEANING
# ==========================================

# 1. Clean Dates
date_cols = ['Date Of Service', 'Pri Ins Adjudication Date', 'Pat Last Bill Date', 'Posting Date']
for col in date_cols:
    if col in charges.columns:
        charges[col] = pd.to_datetime(charges[col], errors='coerce')

# 2. Clean Currency (Robust Function)
def clean_currency(x):
    if isinstance(x, str):
        x = x.replace('$', '').replace(',', '').strip()
        if '(' in x and ')' in x: x = '-' + x.replace('(', '').replace(')', '')
    return pd.to_numeric(x, errors='coerce')

money_cols = [
    'Service Charge Amount', 'Allowed Amount', 'Expected Amount',
    'Pri Ins Insurance Payment', 'Pri Ins Insurance Contract Adjustment',
    'Sec Ins Insurance Payment', 'Sec Ins Insurance Contract Adjustment',
    'Other Ins Insurance Payment', 'Other Ins Insurance Contract Adjustment',
    'Pat Payment Amount', 'Other Adjustment'
]

# Apply cleaning to columns that exist
for col in money_cols:
    if col in charges.columns:
        charges[col] = charges[col].apply(clean_currency).fillna(0)

# Safety: Rename 'claim ID' to 'Claim ID' if inconsistent
if 'claim ID' in charges.columns:
    charges.rename(columns={'claim ID': 'Claim ID'}, inplace=True)

print("Charges Data Cleaned.")

Charges Data Cleaned.


In [ ]:
# 4. FINANCIAL MATH (The "Truth" Calculation)
# ==========================================

# 1. Calculate Total Payments (Cash in Bank)
charges['Total_Payments'] = (
    charges['Pri Ins Insurance Payment'] +
    charges['Sec Ins Insurance Payment'] +
    charges['Other Ins Insurance Payment'] +
    charges['Pat Payment Amount']
)

# 2. Calculate Contract Adjustments (Write-offs)
charges['Total_Contract_Adj'] = (
    charges['Pri Ins Insurance Contract Adjustment'] +
    charges['Sec Ins Insurance Contract Adjustment'] +
    charges['Other Ins Insurance Contract Adjustment']
)

# 3. Calculate Allowed Amount (Billed - Write-offs)
# This overrides Tebra's empty 'Allowed' columns
charges['Calculated_Allowed'] = charges['Service Charge Amount'] - charges['Total_Contract_Adj']
charges['Allowed Amount'] = charges['Calculated_Allowed']

# 4. Calculate Unpaid Balance (The Debt)
charges['Unpaid_Balance'] = charges['Allowed Amount'] - charges['Total_Payments']

# Totals for KPIs
total_billed = charges['Service Charge Amount'].sum()
total_allowed = charges['Allowed Amount'].sum()
total_collected = charges['Total_Payments'].sum()

# KPIs
realization_rate = (total_allowed / total_billed) * 100 if total_billed > 0 else 0
ncr = (total_collected / total_allowed * 100) if total_allowed > 0 else 0

print("--- FINANCIAL HEALTH KPIs ---")
print(f"Realization Rate (Contract vs Billed): {realization_rate:.2f}% (Target: 35-50%)")
print(f"Net Collection Rate (Cash vs Owed):    {ncr:.2f}% (Target: 95%+)")

--- FINANCIAL HEALTH KPIs ---
Realization Rate (Contract vs Billed): 39.47% (Target: 35-50%)
Net Collection Rate (Cash vs Owed):    69.63% (Target: 95%+)


In [ ]:
# 5. WATERFALL & STATUS LOGIC (The "Brain")
# ==========================================

# 1. WATERFALL: Fill missing Payer Names
# If Primary is empty, looks at Secondary, then Other, then defaults to "Unknown/Patient"
charges['Effective_Payer'] = charges['Pri Ins Company Name'].fillna(
    charges['Sec Ins Company Name']
).fillna(
    charges['Other Ins Company Name']
).fillna('Unknown/Patient')

# 2. STATUS CLASSIFICATION: Filter out Pending Claims
# We define "Today" to calculate claim age
analysis_date = pd.to_datetime(datetime.now())
charges['Days_Age'] = (analysis_date - charges['Date Of Service']).dt.days

def classify_claim(row):
    # A. If Balance is effectively zero, it's paid/closed
    if row['Unpaid_Balance'] <= 5:
        return 'Paid / Closed'

    # B. If we billed the patient explicitly (Date exists), it's Patient Debt
    if pd.notnull(row['Pat Last Bill Date']):
        return 'Patient Balance'

    # C. If No Insurance listed, it's Patient (Self-Pay)
    if row['Effective_Payer'] == 'Unknown/Patient':
        return 'Patient Balance'

    # D. ADJUDICATION CHECK (The "Are we sure?" Logic)
    # If Adjudication Date is EMPTY, the insurance hasn't processed it yet.
    if pd.isnull(row['Pri Ins Adjudication Date']):
        if row['Days_Age'] < 45:
            # Young claim, no response yet -> Normal
            return 'Insurance: Pending (Wait)'
        else:
            # Old claim, no response -> CRITICAL PROBLEM
            return 'Insurance: IGNORED (Call)'

    # E. If Adjudicated (Date exists) but Balance remains -> Denial/Underpayment
    return 'Insurance: Processed/Denied'

# Apply the logic
charges['Claim_Status'] = charges.apply(classify_claim, axis=1)
charges['Owed_By'] = charges['Claim_Status'].apply(lambda x: 'Patient' if 'Patient' in x else 'Insurance')

print("Status Classification Complete.")

Status Classification Complete.


In [ ]:
# 6. TRUE DEBT REPORT (The Reality Check)
# ==========================================

# Filter out "Paid" and "Pending" to see only actionable problems
actionable_ar = charges[~charges['Claim_Status'].isin(['Paid / Closed', 'Insurance: Pending (Wait)'])]

summary = actionable_ar.groupby('Claim_Status')['Unpaid_Balance'].sum().reset_index()
summary['% of Problem'] = (summary['Unpaid_Balance'] / summary['Unpaid_Balance'].sum()) * 100

print("--- FINANCIAL REALITY CHECK ---")
print(f"Total 'Paper' AR (Unpaid Balance):  ${charges['Unpaid_Balance'].sum():,.2f}")
print(f"Less: Claims Pending (<45 Days):   -${charges.loc[charges['Claim_Status']=='Insurance: Pending (Wait)', 'Unpaid_Balance'].sum():,.2f}")
print("-" * 40)
print(f"TRUE ACTIONABLE DEBT:               ${summary['Unpaid_Balance'].sum():,.2f}")
print("\nBREAKDOWN OF TRUE DEBT:")
print(summary.to_string(index=False, float_format="${:,.2f}".format))

--- FINANCIAL REALITY CHECK ---
Total 'Paper' AR (Unpaid Balance):  $147,384.25
Less: Claims Pending (<45 Days):   -$461.30
----------------------------------------
TRUE ACTIONABLE DEBT:               $158,512.56

BREAKDOWN OF TRUE DEBT:
               Claim_Status  Unpaid_Balance  % of Problem
  Insurance: IGNORED (Call)       $1,543.40         $0.97
Insurance: Processed/Denied      $20,532.13        $12.95
            Patient Balance     $136,437.03        $86.07


In [ ]:
# 7. OPERATIONAL LIST: INSURANCE "GHOSTS"
# ==========================================
# Claims >45 days old that the insurance has NEVER acknowledged (No Adjudication Date)

ghosts = charges[charges['Claim_Status'] == 'Insurance: IGNORED (Call)'].copy()
ghosts_grouped = ghosts.groupby('Effective_Payer')['Unpaid_Balance'].sum().reset_index().sort_values(by='Unpaid_Balance', ascending=False)

print("\n--- INSURANCE 'GHOST' LIST (Submitted >45 days ago, Ignored) ---")
print(f"Total Value: ${ghosts['Unpaid_Balance'].sum():,.2f}")
print("Top Offenders to Call:")
print(ghosts_grouped.head(10).to_string(index=False, float_format="${:,.2f}".format))


--- INSURANCE 'GHOST' LIST (Submitted >45 days ago, Ignored) ---
Total Value: $1,543.40
Top Offenders to Call:
                        Effective_Payer  Unpaid_Balance
Blue Cross and Blue Shield of Tennessee         $506.24
        United HealthCare of all states         $342.24
                    Amerigroup Tenncare         $158.11
          EmblemHealth Plan, Inc. (MCR)         $113.37
                           Tricare East          $99.60
                                  Cigna          $73.66
                              WellPoint          $67.36
   UHC Community Care for TN (TennCare)          $61.16
         Blue Cross Blue Shield Medigap          $55.56
                                  Aetna          $22.96


In [ ]:
# 8. OPERATIONAL LIST: PATIENT CALL LIST
# ==========================================
# Fresh Patient Debt (0-60 Days) that is worth calling about

call_list = charges[
    (charges['Claim_Status'] == 'Patient Balance') &
    (charges['Days_Age'] <= 60) &
    (charges['Unpaid_Balance'] >= 20)
].copy()

# Sort by Value
call_list = call_list.sort_values(by='Unpaid_Balance', ascending=False)
staff_cols = ['Name', 'Date Of Service', 'Service Charge Amount', 'Total_Payments', 'Unpaid_Balance', 'Days_Age']

print(f"\n--- MONDAY MORNING CALL LIST ({len(call_list)} Patients) ---")
print(f"Total Opportunity: ${call_list['Unpaid_Balance'].sum():,.2f}")
print("Top 10 High-Value Accounts:")
print(call_list[staff_cols].head(10).to_string(index=False, float_format="${:,.2f}".format))

# Optional: call_list[staff_cols].to_csv('Call_List.csv', index=False)


--- MONDAY MORNING CALL LIST (335 Patients) ---
Total Opportunity: $53,117.36
Top 10 High-Value Accounts:
                                                     Name Date Of Service  Service Charge Amount  Total_Payments  Unpaid_Balance  Days_Age
                             Created in Kareo - 12/9/2025      2025-12-12                $590.00           $0.00         $590.00    $21.00
                                                  VACCN 2      2025-12-18                $530.00           $0.00         $530.00    $15.00
                                                    Cigna      2025-12-19                $530.00           $0.00         $530.00    $14.00
                                Patient Intake 2025-08-01      2025-11-24                $530.00           $0.00         $530.00    $39.00
Medicare of Tennessee - Part B - JJ (Palmetto) - Medicare      2025-12-15                $530.00           $0.00         $530.00    $18.00
                                Patient Intake 2025-04-02  

In [ ]:
# 9. AGING REPORT (Dead Debt Analysis)
# ==========================================
# How much of our debt is "Rotten" (>120 Days)?

aging_bins = [0, 30, 60, 90, 120, 9999]
aging_labels = ['0-30 Days', '31-60 Days', '61-90 Days', '91-120 Days', '120+ Days']
charges['Aging_Bucket'] = pd.cut(charges['Days_Age'], bins=aging_bins, labels=aging_labels)

# Filter for actionable debt only (exclude pending/paid)
aging_data = charges[~charges['Claim_Status'].isin(['Paid / Closed', 'Insurance: Pending (Wait)'])]

aging_report = aging_data.groupby(['Aging_Bucket', 'Owed_By'], observed=False)['Unpaid_Balance'].sum().unstack().fillna(0)
aging_report['Total'] = aging_report.sum(axis=1)

print("\n--- AGING REPORT (Actionable Debt Only) ---")
print(aging_report.applymap(lambda x: f"${x:,.2f}"))

rotten_debt = aging_report.loc['120+ Days', 'Total'] if '120+ Days' in aging_report.index else 0
total_ar = aging_report['Total'].sum()
print(f"\n[!] ROTTEN DEBT ALERT: ${rotten_debt:,.2f} ({rotten_debt/total_ar*100:.1f}% of total)")


--- AGING REPORT (Actionable Debt Only) ---
Owed_By        Insurance     Patient       Total
Aging_Bucket                                    
0-30 Days      $1,158.51  $43,012.80  $44,171.31
31-60 Days     $4,654.11  $10,690.61  $15,344.72
61-90 Days     $2,392.72   $6,912.38   $9,305.10
91-120 Days    $1,613.82  $12,523.83  $14,137.65
120+ Days     $12,256.37  $63,297.41  $75,553.78

[!] ROTTEN DEBT ALERT: $75,553.78 (47.7% of total)


C:\Users\jenks\AppData\Local\Temp\ipykernel_49940\1202019812.py:16: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  print(aging_report.applymap(lambda x: f"${x:,.2f}"))


In [ ]:
# 10. DENIAL HUNTER (Coding Issues)
# ==========================================
# Claims that were processed but paid $0 (likely coding errors)

denials = charges[
    (charges['Claim_Status'] == 'Insurance: Processed/Denied') &
    (charges['Total_Payments'] == 0)
].copy()

bad_codes = denials.groupby('Procedure Code').agg({
    'Unpaid_Balance': 'sum',
    'Claim ID': 'count',
    'Effective_Payer': lambda x: x.mode()[0] if not x.mode().empty else 'Various'
}).reset_index().sort_values(by='Unpaid_Balance', ascending=False)

print("\n--- DENIAL HUNTER (Top Procedures with $0 Payment) ---")
print(bad_codes.head(10).to_string(index=False, float_format="${:,.2f}".format))


--- DENIAL HUNTER (Top Procedures with $0 Payment) ---
Procedure Code  Unpaid_Balance  Claim ID                         Effective_Payer
         99204       $4,170.00         9 Blue Cross and Blue Shield of Tennessee
         99213       $2,945.15        14 Blue Cross and Blue Shield of Tennessee
         99203       $1,105.17         5 Blue Cross and Blue Shield of Tennessee
         99396       $1,074.00         3 Blue Cross and Blue Shield of Tennessee
         99215       $1,060.00         2 Blue Cross and Blue Shield of Tennessee
         99214         $958.00         3 Blue Cross and Blue Shield of Tennessee
         99212         $632.33         7 Blue Cross and Blue Shield of Tennessee
         90656         $500.00         5 Blue Cross and Blue Shield of Tennessee
         80305         $435.00         5 Blue Cross and Blue Shield of Tennessee
         99395         $358.00         1         United HealthCare of all states
